# Phase 3 — Embeddings & Retrieval

This notebook converts the semantically chunked employee-policy documents
into vector embeddings and builds a FAISS vector index for semantic retrieval.

Pipeline:

Chunks → Embeddings → FAISS Index → Semantic Retrieval

Notebook structure
```
03 — Embeddings & Retrieval
│
├── 1. Setup & Imports
├── 2. Load Chunk Dataset
├── 3. Inspect Chunk Structure
├── 4. Load Embedding Model
├── 5. Generate Embeddings
├── 6. Normalize Embeddings
├── 7. Build FAISS Index
├── 8. Implement Retrieval
├── 9. Test Individual Queries
├── 10. Batch Retrieval Tests
└── 11. Retrieval Quality Evaluation
```

## Install dependencies

In [2]:
!pip install -q sentence-transformers faiss-cpu

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'E:\\Anaconda3\\envs\\voiceenv\\Lib\\site-packages\\transformers\\models\\muse_glimmer\\video_processing_muse_glimmer.py'
Consider using the `--user` option or check the permissions.



## Imports

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

e:\Anaconda3\envs\voiceenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define project paths

In [2]:
PROJECT_ROOT = Path.cwd().parent

CHUNK_DIR = PROJECT_ROOT/ "Data/chunked_data"

CHUNKS_FILE = CHUNK_DIR / "chunks.jsonl"

print("Project root:", PROJECT_ROOT)
print("Chunks file:", CHUNKS_FILE)

Project root: e:\Projects\Speech AI
Chunks file: e:\Projects\Speech AI\Data\chunked_data\chunks.jsonl


## Load the chunk dataset

In [3]:
chunks = []

with open(
    CHUNKS_FILE,
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        chunks.append(json.loads(line))

print("Total chunks:", len(chunks))

Total chunks: 96


## Convert to DataFrame

In [4]:
chunks_df = pd.DataFrame(chunks)

print("Shape:", chunks_df.shape)

chunks_df.head()

Shape: (96, 10)


,chunk_id,document,file_path,category,section_path,section_title,chunk_index,word_count,character_count,content
0,chunk_000001,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,[Continuing Education],Continuing Education,1,43,243,One of Clef’s core values is “Be better today ...
1,chunk_000002,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,"[Continuing Education, Learning Budget]",Learning Budget,1,174,1038,Every employee has a company budget to support...
2,chunk_000003,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,"[Continuing Education, Mentorship]",Mentorship,1,97,583,Clef understands the value of mentorship. We ...
3,chunk_000004,Continuing Education.md,Benefits and Perks\Continuing Education.md,Benefits and Perks,"[Continuing Education, Speaker Support]",Speaker Support,1,127,747,It makes Clef look great when our employees sp...
4,chunk_000005,Healthcare and Disability Insurance.md,Benefits and Perks\Healthcare and Disability I...,Benefits and Perks,[Healthcare and Disability Insurance],Healthcare and Disability Insurance,1,193,1205,Clef’s priorities with benefits are wellness a...


## Validate the dataset

In [5]:
print("Total chunks:", len(chunks_df))

print(
    "Unique chunk IDs:",
    chunks_df["chunk_id"].nunique()
)

print(
    "Duplicate IDs:",
    chunks_df["chunk_id"].duplicated().sum()
)

print(
    "Empty content:",
    chunks_df["content"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print(
    "Maximum words:",
    chunks_df["word_count"].max()
)

Total chunks: 96
Unique chunk IDs: 96
Duplicate IDs: 0
Empty content: 0
Maximum words: 350


## Load the Embedding Model

Select model

For the first version, use:

In [6]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

print("Embedding model:", MODEL_NAME)

Embedding model: BAAI/bge-small-en-v1.5


This is an English retrieval model and fits your current single-company, English-only project.

## Load model

In [7]:
embedding_model = SentenceTransformer(
    MODEL_NAME
)

print("Embedding model loaded successfully.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1605.73it/s]


Embedding model loaded successfully.


The first run will download the model.

## Generate Embeddings

Prepare text

In [8]:
texts = chunks_df["content"].tolist()

print("Texts prepared:", len(texts))
print("\nFirst text:\n")
print(texts[0])

Texts prepared: 96

First text:

One of Clef’s core values is “Be better today than yesterday,” so it’s important that we support our employees’ efforts to learn, grow, and improve. These are some of the key benefits of working at Clef, and are central to our company culture.


Remember:

We embed the actual chunk content, not the filename or metadata.

The metadata will remain attached to the vector so that we can identify the source after retrieval.

## Generate embeddings

In [9]:
embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches: 100%|██████████| 3/3 [00:15<00:00,  5.06s/it]

Embedding shape: (96, 384)


## Check embeddings

In [10]:
print("Data type:", embeddings.dtype)
print("Shape:", embeddings.shape)
print("Minimum value:", embeddings.min())
print("Maximum value:", embeddings.max())

Data type: float32
Shape: (96, 384)
Minimum value: -0.34674367
Maximum value: 0.5286173


## Normalize Embeddings
We're doing this because we'll use cosine similarity for semantic retrieval.

In [11]:
embeddings = embeddings.astype(
    "float32"
)

faiss.normalize_L2(
    embeddings
)

print("Embeddings normalized.")

Embeddings normalized.


## Build FAISS Index
Create index

IndexFlatIP means Inner Product.

Because we've normalized the vectors, inner product corresponds to cosine similarity.

In [12]:
embedding_dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(
    embedding_dimension
)

print(
    "Embedding dimension:",
    embedding_dimension
)

Embedding dimension: 384


## Add embeddings

In [13]:
index.add(
    embeddings
)

print(
    "Vectors stored in FAISS:",
    index.ntotal
)

Vectors stored in FAISS: 96


## Build the Retrieval Function

In [14]:
def retrieve_chunks(
    query,
    top_k=5
):
    """
    Retrieve the most semantically relevant
    policy chunks for a user query.
    """

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(
        query_embedding
    )

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):

        result = chunks[idx].copy()

        result["similarity_score"] = float(
            score
        )

        results.append(result)

    return results

## First Retrieval Test

In [15]:
query = "How much vacation time do employees get?"

results = retrieve_chunks(
    query,
    top_k=5
)

for rank, result in enumerate(
    results,
    start=1
):

    print("\n" + "=" * 80)

    print(f"Rank: {rank}")
    print(
        f"Similarity: "
        f"{result['similarity_score']:.4f}"
    )

    print(
        f"Document: "
        f"{result['document']}"
    )

    print(
        f"Section: "
        f"{' → '.join(result['section_path'])}"
    )

    print("\nContent:")
    print(result["content"])


Rank: 1
Similarity: 0.7806
Document: Vacation and Sick Leave.md
Section: Vacation and Sick Leave

Content:
Taking time off and recharging is critical to doing your best work at Clef, so in addition to the recognized Holiday List, Clef offers 3 weeks (15 days) of paid vacation every year that accrues 1.25 of a day per month of work. Employees should schedule their vacations, let the rest of the team know, and add it to their shared work calendar at least a week in advance.

Employees also accrue 1 hour of sick leave for every 30 hours of work, but cannot accrue more than 5 days of sick leave.

Employees should report vacation and sick days to the founder they report to, who will mark it in the payroll system (which keeps track of accrued days and should include them on every pay stub).

Employees with chronic or terminal illnesses should talk with the founder they report to about their needs for remote work, flexible time, disability leave and/or other support.

Rank: 2
Similarity: 0.7

### Important
What we're checking

For:

"How much vacation time do employees get?"

we want the Vacation and Sick Leave content to appear near the top.

We're not expecting the LLM to answer yet.

We're only testing:

Question
   ↓
Embedding
   ↓
Semantic Search
   ↓
Correct policy retrieved?

## Test Multiple Employee Questions

In [16]:
test_queries = [
    "How much vacation time do employees get?",
    "Can I work remotely?",
    "What benefits does the company provide?",
    "What is the parental leave policy?",
    "How does the learning budget work?",
    "What are the company's core values?",
    "What should I do if I have a workplace complaint?",
    "What holidays does the company observe?",
    "Can I use company email for personal purposes?",
    "What happens if an employee violates the drug and alcohol policy?"
]

## Run retrieval tests

In [17]:
for query in test_queries:

    results = retrieve_chunks(
        query,
        top_k=3
    )

    print("\n" + "=" * 90)
    print("QUERY:", query)

    for rank, result in enumerate(
        results,
        start=1
    ):

        print(
            f"\n{rank}. "
            f"{result['section_title']}"
        )

        print(
            f"   Score: "
            f"{result['similarity_score']:.4f}"
        )

        print(
            f"   Document: "
            f"{result['document']}"
        )


QUERY: How much vacation time do employees get?

1. Vacation and Sick Leave
   Score: 0.7806
   Document: Vacation and Sick Leave.md

2. Sabbatical
   Score: 0.7076
   Document: Sabbatical.md

3. New Parent Leave
   Score: 0.6736
   Document: New Parent Leave.md

QUERY: Can I work remotely?

1. Extended remote work
   Score: 0.7589
   Document: Working Remotely.md

2. Loss of the privilege
   Score: 0.7583
   Document: Working Remotely.md

3. Plan &amp; Prepare Beforehand
   Score: 0.7512
   Document: Working Remotely.md

QUERY: What benefits does the company provide?

1. Continuing Education
   Score: 0.6640
   Document: Continuing Education.md

2. Approach
   Score: 0.6419
   Document: Working Remotely.md

3. Our mission is to empower everyone to own their identity online.
   Score: 0.6394
   Document: Mission Statement.md

QUERY: What is the parental leave policy?

1. New Parent Leave
   Score: 0.7319
   Document: New Parent Leave.md

2. Pregnancy Disability Leave
   Score: 0.6865


## Create a Better Inspection Function

In [18]:
def show_results(
    query,
    top_k=5
):

    results = retrieve_chunks(
        query,
        top_k=top_k
    )

    print("\nQUERY")
    print("-" * 80)
    print(query)

    for rank, result in enumerate(
        results,
        start=1
    ):

        print("\n" + "=" * 80)

        print(
            f"Rank: {rank}"
        )

        print(
            f"Similarity: "
            f"{result['similarity_score']:.4f}"
        )

        print(
            f"Document: "
            f"{result['document']}"
        )

        print(
            f"Section: "
            f"{' → '.join(result['section_path'])}"
        )

        print(
            f"Word count: "
            f"{result['word_count']}"
        )

        print("\nContent:")
        print(result["content"])

In [19]:
show_results(
    "Can employees take leave for the birth of a child?"
)


QUERY
--------------------------------------------------------------------------------
Can employees take leave for the birth of a child?

Rank: 1
Similarity: 0.7525
Document: New Parent Leave.md
Section: New Parent Leave
Word count: 266

Content:
Clef offers 12 weeks of paid leave for all full time employees, regardless of gender or sexual identity, after the birth or adoption of a child. This time is for the new parent to welcome the newborn or newly adopted child or children into their home and family. The leave should be taken within a year after the birth or adoption of the child.

Employees should give the rest of the team as much notice as possible before they take new parent leave, though there is no requirement for how far in advance notification needs to be given. Parenthood can be unexpected and sensitive, but the more that a team can anticipate the absence, the easier it will be to handle.

Paid time off of any kind, including New Parent Leave, does not accrue additional p

In [34]:
show_results(
    "What should I know about working from home?"
)


QUERY
--------------------------------------------------------------------------------
What should I know about working from home?

Rank: 1
Similarity: 0.7215
Document: Working Remotely.md
Section: Working Remotely → Policies → Extended Remote Work → Plan &amp; Prepare Beforehand
Word count: 101

Content:
It's your responsibility to both make sure you are effective and don't let your teammates down — regardless of the location you work from. This means that you should plan &amp; prepare in your free time before you leave to work remotely.

A non-exhaustive list of things to ensure are in order are:

* You will have a fast, consistent wi-fi connection
* You will have a distraction-free environment to work in
* You will have a quiet, private place to take phone calls and meetings
* You will be able to work a full workday every day you're working remotely

Rank: 2
Similarity: 0.7013
Document: Communication and Transparency.md
Section: Communication and Transparency → Communication → Cale

## Save the FAISS Index
Don't make the notebook regenerate the embeddings every time.

In [20]:
INDEX_FILE = CHUNK_DIR / "faiss_index.bin"

faiss.write_index(
    index,
    str(INDEX_FILE)
)

print(
    "FAISS index saved to:",
    INDEX_FILE
)

FAISS index saved to: e:\Projects\Speech AI\Data\chunked_data\faiss_index.bin


## Save embedding metadata
The FAISS index only stores vectors. We also need to preserve which vector corresponds to which chunk.

In [21]:
METADATA_FILE = CHUNK_DIR / "embedding_metadata.json"

with open(
    METADATA_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        chunks,
        f,
        ensure_ascii=False,
        indent=2
    )

print(
    "Metadata saved to:",
    METADATA_FILE
)

Metadata saved to: e:\Projects\Speech AI\Data\chunked_data\embedding_metadata.json


## Final Validation

In [22]:
print("PHASE 3 VALIDATION")
print("=" * 50)

print(
    "Chunks:",
    len(chunks)
)

print(
    "Embedding shape:",
    embeddings.shape
)

print(
    "FAISS vectors:",
    index.ntotal
)

print(
    "Index dimension:",
    index.d
)

print(
    "Index file exists:",
    INDEX_FILE.exists()
)

print(
    "Metadata file exists:",
    METADATA_FILE.exists()
)

PHASE 3 VALIDATION
Chunks: 96
Embedding shape: (96, 384)
FAISS vectors: 96
Index dimension: 384
Index file exists: True
Metadata file exists: True


## Create a larger evaluation dataset
Use around 30–50 queries, covering different types of employee questions:

In [23]:
evaluation_queries = [
    # Vacation / Leave
    {
        "query": "How much vacation time do employees get?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "Can I take leave after having a baby?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "What holidays does the company observe?",
        "expected_document": "Holiday List.md"
    },

    # Remote Work
    {
        "query": "Can I work from home?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "What are the rules for working remotely?",
        "expected_document": "Working Remotely.md"
    },

    # Learning
    {
        "query": "How does the learning budget work?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Does the company pay for educational activities?",
        "expected_document": "Continuing Education.md"
    },

    # Privacy
    {
        "query": "Can I use company email for personal purposes?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Is my company email private?",
        "expected_document": "Employee Privacy.md"
    },

    # Policies
    {
        "query": "What should I do if I have a workplace complaint?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "What happens if someone violates the drug policy?",
        "expected_document": "Drug and Alcohol Policy.md"
    },

    # Values
    {
        "query": "What are the company's core values?",
        "expected_document": "Clef Values.md"
    },
    {
        "query": "What does Clef value as a company?",
        "expected_document": "Clef Values.md"
    },

    # Benefits
    {
        "query": "What benefits are available to employees?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },

    # Sabbatical
    {
        "query": "When can I take a sabbatical?",
        "expected_document": "Sabbatical.md"
    },

    # Referral
    {
        "query": "Is there a bonus for referring someone?",
        "expected_document": "Referral Bonuses.md"
    },
]

## Quering it for over 100 queries for evaluation

In [24]:
evaluation_queries = [
    # =====================================================
    # VACATION / LEAVE (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "How much vacation time do employees get?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "Can I take leave after having a baby?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "What holidays does the company observe?",
        "expected_document": "Holiday List.md"
    },
    {
        "query": "How many days of PTO do I earn per month?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "I'm not feeling well today, how does sick leave work?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "Is Thanksgiving a day off at Clef?",
        "expected_document": "Holiday List.md"
    },
    {
        "query": "My partner and I are adopting a child — what leave am I entitled to?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "Do fathers get parental leave too or just mothers?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "What happens if a holiday falls on a weekend?",
        "expected_document": "Holiday List.md"
    },
    {
        "query": "I need some time away, what are my options?",
        "expected_document": "Vacation and Sick Leave.md"
    },

    # =====================================================
    # SABBATICAL (Direct, Specific Numbers, Paraphrased)
    # =====================================================
    {
        "query": "How long do I have to work before I'm eligible for a sabbatical?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "What is the duration of the sabbatical at Clef?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "Can I use my sabbatical to start a side business?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "Do I still accrue vacation days while on sabbatical?",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "I've been here 5 years, what's the deal with the long break?",
        "expected_document": "Sabbatical.md"
    },

    # =====================================================
    # CONTINUING EDUCATION (Direct, Specific Numbers, Employee Terminology)
    # =====================================================
    {
        "query": "What is the annual learning budget for employees?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Can I use company money to attend a conference?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Does Clef pay for online courses and books?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "How much does Clef reimburse for speaking at events?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Is there a mentorship program at Clef?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "I joined in June, is my learning budget still $4,000?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "How many hours per week can I spend on learning projects?",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Does the learning budget roll over to the next year?",
        "expected_document": "Continuing Education.md"
    },

    # =====================================================
    # HEALTHCARE & INSURANCE (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What health insurance does Clef offer?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "Does the company cover dental and vision?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "What percentage of health insurance does Clef pay for dependents?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "I already have insurance through my spouse, can I opt out?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "When does my health coverage start after joining?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "Is there life insurance or disability coverage?",
        "expected_document": "Healthcare and Disability Insurance.md"
    },

    # =====================================================
    # SALARY & EQUITY (Direct, Specific Numbers, Employee Terminology)
    # =====================================================
    {
        "query": "How is salary determined at Clef?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "What's the salary for a technical employee with more than 5 years experience?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "How many stock options do new employees receive?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "What is the equity vesting schedule?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "Can I trade $5k of salary for more equity?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "Why doesn't Clef negotiate salaries individually?",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "How much do the founders make?",
        "expected_document": "Salary and Equity Compensation.md"
    },

    # =====================================================
    # REFERRAL BONUSES (Direct, Specific Numbers, Paraphrased)
    # =====================================================
    {
        "query": "How much is the referral bonus?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "If I refer someone who gets hired, when do I get paid?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "What's the process for referring a friend to work at Clef?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "Are founders eligible for the referral bonus?",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "How long does a referred candidate have to be hired within?",
        "expected_document": "Referral Bonuses.md"
    },

    # =====================================================
    # OTHER PROTECTED ABSENCES (Direct, Vague, Related Policies)
    # =====================================================
    {
        "query": "How many days of bereavement leave can I take?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "I got called for jury duty, am I still paid?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "My mother-in-law passed away, can I take time off?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "What is pregnancy disability leave and how long is it?",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "I have a family emergency, what should I do?",
        "expected_document": "Other Protected Absences.md"
    },

    # =====================================================
    # WORKING REMOTELY (Direct, Paraphrased, Employee Terminology)
    # =====================================================
    {
        "query": "What's Clef's policy on working from home?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "Can I work remotely for a whole week?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "Do I need my manager's approval to work remotely for an extended period?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "Does Clef pay for coworking spaces?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "What happens if I'm not performing well while working remotely?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "How far in advance do I need to notify the team about extended remote work?",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "I want to work from my partner's place in another city for a few days — what do I need to do?",
        "expected_document": "Working Remotely.md"
    },

    # =====================================================
    # EMPLOYEE PRIVACY (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "Can the company read my emails?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Does Clef monitor internet usage?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Can my manager search my desk or laptop?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Is it okay to send personal emails from my work account?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "What should I do if I think my computer has a virus?",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "Can I share my work email password with a coworker?",
        "expected_document": "Employee Privacy.md"
    },

    # =====================================================
    # CODE OF CONDUCT (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What behavior is not tolerated in Clef community spaces?",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "Someone is being rude in the Clef Slack channel — what should I do?",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "Does the code of conduct apply to Twitter and Facebook?",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "Who do I contact about harassment in a Clef community?",
        "expected_document": "Code of Conduct in the Community.md"
    },

    # =====================================================
    # COMPLAINT POLICY (Direct, Vague, Related Policies)
    # =====================================================
    {
        "query": "How do I file a workplace complaint at Clef?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "Will I face retaliation for reporting a problem?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "What happens after I submit a complaint?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "What if my complaint is about one of the founders?",
        "expected_document": "Complaint Policy.md"
    },

    # =====================================================
    # DRUG & ALCOHOL POLICY (Direct, Vague)
    # =====================================================
    {
        "query": "Is alcohol allowed in the office?",
        "expected_document": "Drug and Alcohol Policy.md"
    },
    {
        "query": "What is Clef's policy on recreational drug use?",
        "expected_document": "Drug and Alcohol Policy.md"
    },
    {
        "query": "Can we have beer at a company celebration?",
        "expected_document": "Drug and Alcohol Policy.md"
    },

    # =====================================================
    # AT-WILL EMPLOYMENT (Direct, Paraphrased)
    # =====================================================
    {
        "query": "Can Clef fire me without a reason?",
        "expected_document": "At-Will Employment.md"
    },
    {
        "query": "What does at-will employment mean at Clef?",
        "expected_document": "At-Will Employment.md"
    },
    {
        "query": "Who can change my at-will employment status?",
        "expected_document": "At-Will Employment.md"
    },

    # =====================================================
    # EQUAL OPPORTUNITY EMPLOYMENT (Direct, Related Policies)
    # =====================================================
    {
        "query": "Does Clef discriminate based on gender or race?",
        "expected_document": "Equal Opportunity Employment.md"
    },
    {
        "query": "What is Clef's stance on diversity in hiring?",
        "expected_document": "Equal Opportunity Employment.md"
    },

    # =====================================================
    # CLEF VALUES (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What are Clef's core values?",
        "expected_document": "Clef Values.md"
    },
    {
        "query": "What does 'be better today than yesterday' mean?",
        "expected_document": "Clef Values.md"
    },
    {
        "query": "How does Clef think about inclusion?",
        "expected_document": "Clef Values.md"
    },
    {
        "query": "Why does Clef emphasize trust?",
        "expected_document": "Clef Values.md"
    },

    # =====================================================
    # MISSION STATEMENT (Direct, Vague)
    # =====================================================
    {
        "query": "What is Clef's mission?",
        "expected_document": "Mission Statement.md"
    },
    {
        "query": "What problem is Clef trying to solve?",
        "expected_document": "Mission Statement.md"
    },

    # =====================================================
    # ONBOARDING / WELCOME (Direct, Employee Terminology, Vague)
    # =====================================================
    {
        "query": "What should I expect on my first day at Clef?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "What are the typical working hours at Clef?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "How does Clef celebrate a new employee's first day?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "I just got hired, where do I start?",
        "expected_document": "Welcome to Clef.md"
    },
    {
        "query": "What is the goal for my first day at work?",
        "expected_document": "Welcome to Clef.md"
    },

    # =====================================================
    # HANDBOOK INTRODUCTION (Direct)
    # =====================================================
    {
        "query": "Who is the CEO of Clef?",
        "expected_document": "Handbook Introduction.md"
    },
    {
        "query": "What is the purpose of the employee handbook?",
        "expected_document": "Handbook Introduction.md"
    },

    # =====================================================
    # ONE ON ONES (Direct, Paraphrased, Employee Terminology)
    # =====================================================
    {
        "query": "How often are one-on-one meetings held?",
        "expected_document": "One on Ones.md"
    },
    {
        "query": "Who sets the agenda for 1:1 meetings?",
        "expected_document": "One on Ones.md"
    },
    {
        "query": "What's the minimum duration of a one on one?",
        "expected_document": "One on Ones.md"
    },

    # =====================================================
    # OBJECTIVES AND KEY RESULTS (Direct, Specific Numbers, Employee Terminology)
    # =====================================================
    {
        "query": "How are OKRs scored at Clef?",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "How often do we set OKRs?",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "Are OKRs used for performance reviews?",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "What is a good OKR score?",
        "expected_document": "Objectives and Key Results.md"
    },

    # =====================================================
    # COMMUNICATION & TRANSPARENCY (Direct, Paraphrased)
    # =====================================================
    {
        "query": "What happens on Fridays in terms of team updates?",
        "expected_document": "Communication and Transparency.md"
    },
    {
        "query": "How should I use Slack when working remotely?",
        "expected_document": "Communication and Transparency.md"
    },
    {
        "query": "Should conversations happen in public or private Slack channels?",
        "expected_document": "Communication and Transparency.md"
    },

    # =====================================================
    # DIRECT REPORTS (Direct)
    # =====================================================
    {
        "query": "Who do employees report to at Clef?",
        "expected_document": "Direct Reports.md"
    },

    # =====================================================
    # PRODUCT MANIFESTO (Direct, Paraphrased, Vague)
    # =====================================================
    {
        "query": "What is Clef's core product value?",
        "expected_document": "Product Manifesto.md"
    },
    {
        "query": "Why do people love using Clef over other login solutions?",
        "expected_document": "Product Manifesto.md"
    },

    # =====================================================
    # OPERATIONS: BUDGETING, HACK WEEKS, SHARING FILES, EFFECTIVE MEETINGS
    # =====================================================
    {
        "query": "How is company spending organized at Clef?",
        "expected_document": "Budgeting.md"
    },
    {
        "query": "What are the budget categories at Clef?",
        "expected_document": "Budgeting.md"
    },
    {
        "query": "What is a hack week and when does it happen?",
        "expected_document": "Hack Weeks.md"
    },
    {
        "query": "Can I work on personal projects during hack week?",
        "expected_document": "Hack Weeks.md"
    },
    {
        "query": "How should files and projects be organized at Clef?",
        "expected_document": "Sharing Files.md"
    },
    {
        "query": "What are the base directories every Clef employee should have?",
        "expected_document": "Sharing Files.md"
    },
    {
        "query": "What are the meeting time requirements at Clef?",
        "expected_document": "Effective Meetings.md"
    },
    {
        "query": "Do meetings need to have a video call option?",
        "expected_document": "Effective Meetings.md"
    },

    # =====================================================
    # POLICY CHANGES (Direct, Paraphrased)
    # =====================================================
    {
        "query": "How are policy changes proposed and adopted at Clef?",
        "expected_document": "Policy Changes.md"
    },
    {
        "query": "What tools does Clef use for policy discussions?",
        "expected_document": "Policy Changes.md"
    },
    # =====================================================
    # CROSS-DOCUMENT / SEMANTICALLY SIMILAR DOCUMENT QUERIES
    # These are tricky — two docs could plausibly match
    # =====================================================
    {
        "query": "What kind of time off can I get after becoming a new parent?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "How does taking parental leave affect my sabbatical eligibility?",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "What's the difference between sick leave and vacation?",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "If something bad happens at work who should I talk to — my manager or HR?",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "Does the code of conduct apply inside the office or only online?",
        "expected_document": "Code of Conduct in the Community.md"
    },
]


## Evaluate Top-K retrieval

Use your existing retrieval function. If your current function is called retrieve_chunks(), use:

In [25]:
def evaluate_retrieval(
    evaluation_queries,
    k=5
):
    results = []

    for item in evaluation_queries:

        query = item["query"]
        expected = item["expected_document"]

        retrieved = retrieve_chunks(
            query,
            top_k=k
        )

        retrieved_documents = [
            r["document"]
            for r in retrieved
        ]

        results.append({
            "query": query,
            "expected": expected,
            "retrieved": retrieved_documents,
            "top_1": (
                expected == retrieved_documents[0]
                if retrieved_documents
                else False
            ),
            "top_3": (
                expected in retrieved_documents[:3]
            ),
            "top_5": (
                expected in retrieved_documents[:5]
            )
        })

    return results

Important: if your existing retrieval function has a different name or returns a different structure, adapt those two parts rather than rewriting your embedding system.

## Run the evaluation

In [26]:
evaluation_results = evaluate_retrieval(
    evaluation_queries,
    k=5
)

In [27]:
import pandas as pd

evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

,query,expected,retrieved,top_1,top_3,top_5
0,How much vacation time do employees get?,Vacation and Sick Leave.md,"[Vacation and Sick Leave.md, Sabbatical.md, Ne...",True,True,True
1,Can I take leave after having a baby?,New Parent Leave.md,"[New Parent Leave.md, Other Protected Absences...",True,True,True
2,What holidays does the company observe?,Holiday List.md,"[Holiday List.md, Communication and Transparen...",True,True,True
3,How many days of PTO do I earn per month?,Vacation and Sick Leave.md,"[Vacation and Sick Leave.md, Other Protected A...",True,True,True
4,"I'm not feeling well today, how does sick leav...",Vacation and Sick Leave.md,"[Vacation and Sick Leave.md, Other Protected A...",True,True,True
...,...,...,...,...,...,...
111,What kind of time off can I get after becoming...,New Parent Leave.md,"[New Parent Leave.md, Other Protected Absences...",True,True,True
112,How does taking parental leave affect my sabba...,New Parent Leave.md,"[New Parent Leave.md, Sabbatical.md, Other Pro...",True,True,True
113,What's the difference between sick leave and v...,Vacation and Sick Leave.md,"[Vacation and Sick Leave.md, New Parent Leave....",True,True,True
114,If something bad happens at work who should I ...,Complaint Policy.md,"[One on Ones.md, One on Ones.md, Effective Mee...",False,False,False


## Calculate the actual metrics

In [28]:
total = len(evaluation_df)

top_1_accuracy = (
    evaluation_df["top_1"].sum() / total
)

top_3_recall = (
    evaluation_df["top_3"].sum() / total
)

top_5_recall = (
    evaluation_df["top_5"].sum() / total
)

print(f"Total queries: {total}")

print(
    f"Top-1 Accuracy: "
    f"{top_1_accuracy:.2%}"
)

print(
    f"Top-3 Recall: "
    f"{top_3_recall:.2%}"
)

print(
    f"Top-5 Recall: "
    f"{top_5_recall:.2%}"
)

Total queries: 116
Top-1 Accuracy: 78.45%
Top-3 Recall: 87.93%
Top-5 Recall: 89.66%


## Find the failures

In [29]:
failures = evaluation_df[
    ~evaluation_df["top_3"]
]

failures[
    [
        "query",
        "expected",
        "retrieved"
    ]
]

,query,expected,retrieved
5,Is Thanksgiving a day off at Clef?,Holiday List.md,"[Vacation and Sick Leave.md, Other Protected A..."
9,"I need some time away, what are my options?",Vacation and Sick Leave.md,"[Communication and Transparency.md, Working Re..."
70,Can Clef fire me without a reason?,At-Will Employment.md,"[Complaint Policy.md, Employee Privacy.md, Emp..."
75,What are Clef's core values?,Clef Values.md,"[Welcome to Clef.md, Continuing Education.md, ..."
77,How does Clef think about inclusion?,Clef Values.md,"[Equal Opportunity Employment.md, Welcome to C..."
78,Why does Clef emphasize trust?,Clef Values.md,"[Welcome to Clef.md, One on Ones.md, Employee ..."
79,What is Clef's mission?,Mission Statement.md,"[Welcome to Clef.md, Welcome to Clef.md, Handb..."
80,What problem is Clef trying to solve?,Mission Statement.md,"[Complaint Policy.md, Product Manifesto.md, We..."
99,What is Clef's core product value?,Product Manifesto.md,"[Welcome to Clef.md, Welcome to Clef.md, Equal..."
101,How is company spending organized at Clef?,Budgeting.md,"[Continuing Education.md, Continuing Education..."


In [30]:
import json
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------
# Save retrieval evaluation results
# ---------------------------------------------------------

EVALUATION_DIR = Path("evaluation_results")
EVALUATION_DIR.mkdir(exist_ok=True)

evaluation_output = {
    "evaluation_date": datetime.now().isoformat(),
    "total_queries": len(evaluation_results),
    "results": evaluation_results
}

output_file = EVALUATION_DIR / "retrieval_evaluation.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Evaluation saved to: {output_file}")
print(f"Total queries saved: {len(evaluation_results)}")

Evaluation saved to: evaluation_results\retrieval_evaluation.json
Total queries saved: 116


In [31]:
evaluation_output = {
    "evaluation_date": datetime.now().isoformat(),
    "total_queries": len(evaluation_df),
    "results": evaluation_df.to_dict(orient="records")
}

output_file = EVALUATION_DIR / "retrieval_evaluation.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved: {output_file}")

Saved: evaluation_results\retrieval_evaluation.json


In [32]:
evaluation_queries_long = [
    # =====================================================
    # VACATION / LEAVE — Long-form queries
    # =====================================================
    {
        "query": "I've been working at Clef for about six months now and I'm wondering how many vacation days I've accumulated so far and whether unused sick days carry over to the next year",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "My spouse and I are expecting a baby in a few months and I want to understand the full maternity and paternity leave policy including how it interacts with California state disability programs",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "I recently adopted a child and I'm trying to figure out within what timeframe I need to take my new parent leave and whether I need to give any advance notice to my team",
        "expected_document": "New Parent Leave.md"
    },
    {
        "query": "I have a chronic illness that sometimes makes it hard to come into the office regularly — is there any policy around flexible work arrangements or disability support for someone in my situation",
        "expected_document": "Vacation and Sick Leave.md"
    },
    {
        "query": "If I take the full twelve weeks of new parent leave does that time count toward my five year requirement for becoming eligible for a sabbatical or does it pause the clock",
        "expected_document": "New Parent Leave.md"
    },

    # =====================================================
    # SABBATICAL — Long-form queries
    # =====================================================
    {
        "query": "I've been at Clef for almost five years and I'm interested in taking a sabbatical to volunteer at a nonprofit — do I need to present something to the team when I return and how far in advance should I notify everyone",
        "expected_document": "Sabbatical.md"
    },
    {
        "query": "I'm worried that if I take a three month sabbatical I might lose my accrued sick days — can you explain whether paid time off continues to accumulate during the sabbatical period or if it freezes",
        "expected_document": "Sabbatical.md"
    },

    # =====================================================
    # CONTINUING EDUCATION — Long-form queries
    # =====================================================
    {
        "query": "I want to attend a machine learning conference in Europe next quarter — would the flight and hotel costs come out of my learning budget or is there a separate travel budget for professional development events",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "I've been invited to speak at a cybersecurity conference and I'm wondering how much Clef will cover for travel expenses and whether this comes from the same pool as the annual learning budget or a different one",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "I started working at Clef in July of this year and I want to know if my learning budget is the full four thousand dollars or if it is reduced since I joined after the midpoint of the calendar year",
        "expected_document": "Continuing Education.md"
    },
    {
        "query": "Can I use a portion of my work hours during the week to study for a certification that is related to my role at Clef and if so is there a limit on how many hours I can dedicate to that",
        "expected_document": "Continuing Education.md"
    },

    # =====================================================
    # HEALTHCARE & INSURANCE — Long-form queries
    # =====================================================
    {
        "query": "My wife already has health insurance through her employer and I don't need Clef's medical plan — is there any financial incentive or allowance if I choose to waive my medical coverage through TriNet",
        "expected_document": "Healthcare and Disability Insurance.md"
    },
    {
        "query": "I'm starting at Clef next month and I want to add my two kids to my health insurance plan — what percentage of the dependent coverage does the company contribute and when does the coverage actually begin",
        "expected_document": "Healthcare and Disability Insurance.md"
    },

    # =====================================================
    # SALARY & EQUITY — Long-form queries
    # =====================================================
    {
        "query": "I'm a software engineer with seven years of experience and I'd like to understand what my starting salary would be at Clef and whether there is room for individual negotiation beyond the standard rubric",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "I noticed the equity vesting schedule at Clef is six years instead of the typical four — can you explain the reasoning behind that and how many stock options a new employee typically receives",
        "expected_document": "Salary and Equity Compensation.md"
    },
    {
        "query": "If I choose to take the five thousand dollar salary reduction in exchange for additional equity how many extra stock options would I receive and what total percentage of the company would that represent",
        "expected_document": "Salary and Equity Compensation.md"
    },

    # =====================================================
    # REFERRAL BONUSES — Long-form queries
    # =====================================================
    {
        "query": "I referred a friend to Clef about four months ago and she just got an offer — do I qualify for the referral bonus and when exactly would I expect to receive the payment after she starts",
        "expected_document": "Referral Bonuses.md"
    },
    {
        "query": "Two people on the team both claim they referred the same candidate independently — how does the company determine who gets the five thousand dollar referral bonus in a situation like this",
        "expected_document": "Referral Bonuses.md"
    },

    # =====================================================
    # WORKING REMOTELY — Long-form queries
    # =====================================================
    {
        "query": "I'm planning to work from my parents house in another state for about ten days next month — do I need to get my manager's approval beforehand and how much advance notice should I give the rest of the team",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "I've been working remotely a couple days a week but my manager says my productivity has dropped — can they revoke my remote work privileges entirely and what process has to happen before that decision is made",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "When I'm working from home I know I should be available on Slack but I'm unclear about the exact expectations — is there a specific set of hours where I need to be reachable for meetings and collaboration",
        "expected_document": "Working Remotely.md"
    },
    {
        "query": "I want to work from a coffee shop while traveling but I'm concerned about the internet connection requirements — what are the specific prerequisites Clef expects me to have in place before working from somewhere irregular",
        "expected_document": "Working Remotely.md"
    },

    # =====================================================
    # EMPLOYEE PRIVACY — Long-form queries
    # =====================================================
    {
        "query": "I sometimes send personal emails from my Clef email address and I want to know if the company has the ability to read those messages even if I mark them as private or personal in the subject line",
        "expected_document": "Employee Privacy.md"
    },
    {
        "query": "I keep a personal journal on my work laptop and I want to know if management has the right to access files stored on company devices or if my personal data on company property is protected",
        "expected_document": "Employee Privacy.md"
    },

    # =====================================================
    # CODE OF CONDUCT — Long-form queries
    # =====================================================
    {
        "query": "A community member made an offensive joke in the Clef Slack channel and someone complained about it — what does the code of conduct say about jokes and what are the consequences for the person who made the comment",
        "expected_document": "Code of Conduct in the Community.md"
    },
    {
        "query": "I witnessed someone being harassed at a Clef-hosted event last week and I want to report it — who exactly should I contact and does it matter if the harassment was not directed at me personally",
        "expected_document": "Code of Conduct in the Community.md"
    },

    # =====================================================
    # COMPLAINT POLICY — Long-form queries
    # =====================================================
    {
        "query": "I want to file a complaint about something that happened at work but I'm afraid my manager might treat me differently afterward — does Clef have a policy against retaliation for employees who report issues",
        "expected_document": "Complaint Policy.md"
    },
    {
        "query": "If my complaint involves one of the founders of the company how is the investigation handled differently and is there a possibility that an outside investigator would be brought in to ensure impartiality",
        "expected_document": "Complaint Policy.md"
    },

    # =====================================================
    # OKRs — Long-form queries
    # =====================================================
    {
        "query": "I'm new to the OKR system and I don't understand how scoring works at the end of the quarter — if I score a perfect ten on all my key results does that mean I set my goals too low or is that considered ideal",
        "expected_document": "Objectives and Key Results.md"
    },
    {
        "query": "I want to set a key result that involves delivering a project by a certain date — how should I structure the scoring so that points are deducted fairly if the project ends up being delivered late",
        "expected_document": "Objectives and Key Results.md"
    },

    # =====================================================
    # ONE ON ONES — Long-form queries
    # =====================================================
    {
        "query": "I have a problem with a coworker and I want to bring it up during my one on one meeting — will my manager step in to address it directly or do they need my permission before talking to that person about it",
        "expected_document": "One on Ones.md"
    },
    {
        "query": "My one on one meetings with my manager feel more like status updates than meaningful conversations — what does Clef recommend regarding the agenda and tone of these meetings to make them more productive",
        "expected_document": "One on Ones.md"
    },

    # =====================================================
    # COMMUNICATION & TRANSPARENCY — Long-form queries
    # =====================================================
    {
        "query": "When I set my Slack status to Do Not Disturb and the indicator shows green what does that signal to my teammates — should they expect a response immediately or should they wait until my focus time is over",
        "expected_document": "Communication and Transparency.md"
    },
    {
        "query": "I've noticed that a lot of important decisions are being discussed in private Slack channels and I feel out of the loop — what is Clef's official guideline on using public versus private channels for team conversations",
        "expected_document": "Communication and Transparency.md"
    },

    # =====================================================
    # OPERATIONS — Long-form queries
    # =====================================================
    {
        "query": "I have an idea for a hack week project that would require three other people to help me build it — how do I pitch this project to the team and are there any restrictions on what we can work on during that week",
        "expected_document": "Hack Weeks.md"
    },
    {
        "query": "When a project is completed and no longer active how should I archive the project folder and what naming convention does Clef use so that old projects are easy to find later on",
        "expected_document": "Sharing Files.md"
    },
    {
        "query": "I need to schedule a meeting with a remote colleague but they are in a different time zone — what are the core hours during which all Clef employees are expected to be available for face to face meetings",
        "expected_document": "Effective Meetings.md"
    },

    # =====================================================
    # POLICY CHANGES — Long-form queries
    # =====================================================
    {
        "query": "I disagree with a recent policy change that was merged into the handbook and I want to voice my concerns — what is the proper channel for giving feedback and will there be a team discussion about it before it takes effect",
        "expected_document": "Policy Changes.md"
    },
    {
        "query": "I heard that employees need to sign an acknowledgement every time the handbook is updated — how often do these updates get adopted and is there a formal review process before changes become official policy",
        "expected_document": "Policy Changes.md"
    },

    # =====================================================
    # CROSS-DOCUMENT / TRICKY — Long-form queries
    # =====================================================
    {
        "query": "I'm pregnant and dealing with severe morning sickness that makes it hard to come to work every day — can I take intermittent pregnancy disability leave while still keeping my job and benefits at Clef",
        "expected_document": "Other Protected Absences.md"
    },
    {
        "query": "I want to understand how Clef avoids salary bias during the hiring process — is there a standard pay scale for all roles or does compensation depend on individual negotiation skills and previous salary history",
        "expected_document": "Salary and Equity Compensation.md"
    },
]


In [33]:
def evaluate_retrieval(
    evaluation_queries_long,
    k=5
):
    results = []

    for item in evaluation_queries_long:

        query = item["query"]
        expected = item["expected_document"]

        retrieved = retrieve_chunks(
            query,
            top_k=k
        )

        retrieved_documents = [
            r["document"]
            for r in retrieved
        ]

        results.append({
            "query": query,
            "expected": expected,
            "retrieved": retrieved_documents,
            "top_1": (
                expected == retrieved_documents[0]
                if retrieved_documents
                else False
            ),
            "top_3": (
                expected in retrieved_documents[:3]
            ),
            "top_5": (
                expected in retrieved_documents[:5]
            )
        })

    return results

Important: if your existing retrieval function has a different name or returns a different structure, adapt those two parts rather than rewriting your embedding system.

## Run the evaluation

In [34]:
evaluation_results = evaluate_retrieval(
    evaluation_queries_long,
    k=5
)

In [35]:
import pandas as pd

evaluation_df = pd.DataFrame(
    evaluation_results
)

evaluation_df

,query,expected,retrieved,top_1,top_3,top_5
0,I've been working at Clef for about six months...,Vacation and Sick Leave.md,"[Vacation and Sick Leave.md, Sabbatical.md, Ot...",True,True,True
1,My spouse and I are expecting a baby in a few ...,New Parent Leave.md,"[Other Protected Absences.md, New Parent Leave...",False,True,True
2,I recently adopted a child and I'm trying to f...,New Parent Leave.md,"[New Parent Leave.md, Working Remotely.md, Com...",True,True,True
3,I have a chronic illness that sometimes makes ...,Vacation and Sick Leave.md,"[Working Remotely.md, Other Protected Absences...",False,False,False
4,If I take the full twelve weeks of new parent ...,New Parent Leave.md,"[New Parent Leave.md, Sabbatical.md, Other Pro...",True,True,True
5,I've been at Clef for almost five years and I'...,Sabbatical.md,"[Sabbatical.md, Working Remotely.md, Welcome t...",True,True,True
6,I'm worried that if I take a three month sabba...,Sabbatical.md,"[Sabbatical.md, Vacation and Sick Leave.md, Ne...",True,True,True
7,I want to attend a machine learning conference...,Continuing Education.md,"[Continuing Education.md, Continuing Education...",True,True,True
8,I've been invited to speak at a cybersecurity ...,Continuing Education.md,"[Continuing Education.md, Continuing Education...",True,True,True
9,I started working at Clef in July of this year...,Continuing Education.md,"[Continuing Education.md, Continuing Education...",True,True,True


## Calculate the actual metrics

In [36]:
total = len(evaluation_df)

top_1_accuracy = (
    evaluation_df["top_1"].sum() / total
)

top_3_recall = (
    evaluation_df["top_3"].sum() / total
)

top_5_recall = (
    evaluation_df["top_5"].sum() / total
)

print(f"Total queries: {total}")

print(
    f"Top-1 Accuracy: "
    f"{top_1_accuracy:.2%}"
)

print(
    f"Top-3 Recall: "
    f"{top_3_recall:.2%}"
)

print(
    f"Top-5 Recall: "
    f"{top_5_recall:.2%}"
)

Total queries: 41
Top-1 Accuracy: 87.80%
Top-3 Recall: 95.12%
Top-5 Recall: 95.12%


## Find the failures

In [37]:
failures = evaluation_df[
    ~evaluation_df["top_3"]
]

failures[
    [
        "query",
        "expected",
        "retrieved"
    ]
]

,query,expected,retrieved
3,I have a chronic illness that sometimes makes ...,Vacation and Sick Leave.md,"[Working Remotely.md, Other Protected Absences..."
10,Can I use a portion of my work hours during th...,Continuing Education.md,"[Working Remotely.md, Welcome to Clef.md, Vaca..."


In [38]:
import json
from pathlib import Path
from datetime import datetime

# ---------------------------------------------------------
# Save retrieval evaluation results 
# ---------------------------------------------------------

EVALUATION_DIR = Path("evaluation_results")
EVALUATION_DIR.mkdir(exist_ok=True)

evaluation_output = {
    "evaluation_date": datetime.now().isoformat(),
    "total_queries": len(evaluation_results),
    "results": evaluation_results
}

output_file = EVALUATION_DIR / "retrieval_evaluation_long.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Evaluation saved to: {output_file}")
print(f"Total queries saved: {len(evaluation_results)}")

Evaluation saved to: evaluation_results\retrieval_evaluation_long.json
Total queries saved: 41


In [39]:
evaluation_output = {
    "evaluation_date": datetime.now().isoformat(),
    "total_queries": len(evaluation_df),
    "results": evaluation_df.to_dict(orient="records")
}

output_file = EVALUATION_DIR / "retrieval_evaluation_long.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        evaluation_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved: {output_file}")

Saved: evaluation_results\retrieval_evaluation_long.json
